# 05 | Ridge regression baseline

**Goal:** a regularised linear benchmark. Ridge minimises squared error plus an L2 penalty on coefficients, which stabilises correlated morphology predictors; features are standardised first because the penalty acts on coefficient size.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'; RESULTS = ROOT / 'results'
import sys; sys.path.insert(0, str(ROOT / 'src'))
df = pd.read_csv(DATA / 'layout_level_dataset.csv')
target = 'target_U_canopy_mean_m_s'
allm = pd.read_csv(RESULTS/'metrics_all_protocols.csv')
cols = ['protocol','model','mae_m_s','rmse_m_s','normalized_rmse_target_range','r2','mae_ci_low','mae_ci_high']
r = allm[(allm.feature_set=='engineered') & allm.model.isin(['Mean baseline','Ridge Regression'])]
print(r[cols].round(3).to_string(index=False))
print('\nConfiguration: median imputation; standard scaling; one-hot categories; Ridge(alpha=1.0).')

    protocol            model  mae_m_s  rmse_m_s  normalized_rmse_target_range     r2  mae_ci_low  mae_ci_high
family_5fold    Mean baseline    0.803     1.017                         0.236 -0.033       0.571        1.099
family_5fold Ridge Regression    0.403     0.510                         0.118  0.740       0.313        0.504
  family_loo    Mean baseline    0.841     1.058                         0.245 -0.118       0.593        1.129
  family_loo Ridge Regression    0.405     0.514                         0.119  0.736       0.316        0.508
layout_5fold    Mean baseline    0.795     1.000                         0.232 -0.000       0.573        1.066
layout_5fold Ridge Regression    0.333     0.424                         0.099  0.820       0.266        0.410

Configuration: median imputation; standard scaling; one-hot categories; Ridge(alpha=1.0).


## Effect of the engineered features on Ridge

In [2]:
b = allm[(allm.feature_set=='baseline_features') & (allm.model=='Ridge Regression')][['protocol','mae_m_s','r2']].set_index('protocol')
e = allm[(allm.feature_set=='engineered') & (allm.model=='Ridge Regression')][['protocol','mae_m_s','r2']].set_index('protocol')
print(pd.concat({'first-submission features': b, 'engineered features': e}, axis=1).round(3).to_string())

             first-submission features        engineered features       
                               mae_m_s     r2             mae_m_s     r2
protocol                                                                
family_5fold                     0.442  0.653               0.403  0.740
family_loo                       0.514  0.553               0.405  0.736
layout_5fold                     0.402  0.720               0.333  0.820


Under family-held-out validation Ridge reaches MAE about 0.40 m/s (R² about 0.74), better than the mean baseline but far from the tree models. The density-squared term is what helps the linear model: the decay is convex, which a straight line cannot follow. Coefficients are conditional associations, not causal effects.